# SegFormer MiT-B2 training on 1,000 GCS tamper examples

This notebook samples **1,000 random base screenshot IDs** from:

`gs://maverick91/audit_finding_project/screenshot_tamper_final/`

For each sampled base image it picks one generated tampered image + mask pair, records the IDs, creates an exact **70/15/15** split, and runs **three learning-rate trials** for SegFormer MiT-B2. Each trial trains for up to **50 epochs** and uses validation Dice early stopping with **patience/tolerance = 3 epochs**.

Outputs written locally:

- `outputs/sampled_1000_splits.csv` - all selected IDs and train/val/test split
- `outputs/trial_history.csv` - per-epoch train/validation metrics for all LR trials
- `outputs/trial_summary.csv` - best epoch and validation metrics per trial
- `outputs/best_trial_summary.json` - selected best LR/trial/epoch
- `outputs/test_metrics.json` - held-out test metrics from the best checkpoint
- `outputs/checkpoints/.../best.pt` - best checkpoint for each trial
- `outputs/best_model_test_predictions.png` - qualitative test predictions

> Security note: do **not** paste service-account private keys into shared notebooks. Use Colab Secrets, Kaggle Secrets, an uploaded JSON file, or `GOOGLE_APPLICATION_CREDENTIALS`.

## 1. Install dependencies

Run this once at the start of a Colab/Kaggle session. If your runtime already has the packages, this will be quick.

In [ ]:
%pip install -q segmentation-models-pytorch albumentations opencv-python-headless google-cloud-storage pandas scikit-learn matplotlib tqdm

## 2. Imports and configuration

Update `LEARNING_RATE_TRIALS`, `BATCH_SIZE`, or `IMAGE_SIZE` if your GPU memory requires it. The default three LR trials are intentionally conservative for fine-tuning a pretrained transformer backbone on a small subset.

In [ ]:
import json
import math
import os
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import segmentation_models_pytorch as smp
from google.cloud import storage

# -------------------------
# Reproducibility / sampling
# -------------------------
SEED = 42
SAMPLE_SIZE = 1000
SPLIT_RATIOS = (0.70, 0.15, 0.15)

# Sample one tampered image per base screenshot manifest. This gives exactly
# 1,000 image/mask training examples and avoids multiple alterations from the
# same base screenshot leaking across splits.
SAMPLE_MODE = "one_tamper_per_base_image"

# -------------------------
# GCS source
# -------------------------
GCS_BUCKET = "maverick91"
GCS_PREFIX = "audit_finding_project/screenshot_tamper_final"
MANIFEST_PREFIX = f"{GCS_PREFIX}/manifests"
IMAGE_PREFIX = f"{GCS_PREFIX}/images"

# -------------------------
# Training config
# -------------------------
ENCODER_NAME = "mit_b2"
ENCODER_WEIGHTS = "imagenet"
IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 2
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 3  # tolerance of 3 non-improving validation epochs
MIN_DELTA = 1e-4
LEARNING_RATE_TRIALS = [1e-5, 1e-4, 3e-4]
WEIGHT_DECAY = 1e-5
MIXED_PRECISION = True
THRESHOLD = 0.5

# -------------------------
# Local output/cache paths
# -------------------------
try:
    from google.colab import drive  # noqa: F401
    DEFAULT_ROOT = Path("/content/segformer_gcs_1000")
except Exception:
    DEFAULT_ROOT = Path.cwd() / "segformer_gcs_1000"

RUN_NAME = datetime.utcnow().strftime("segformer_mit_b2_1000_%Y%m%d_%H%M%S")
WORK_DIR = DEFAULT_ROOT
CACHE_DIR = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

for path in [WORK_DIR, CACHE_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Optional: set to True if your service account has write access and you want
# the notebook outputs uploaded back to the bucket.
UPLOAD_OUTPUTS_TO_GCS = False
OUTPUT_GCS_PREFIX = f"{GCS_PREFIX}/segformer_results/{RUN_NAME}"

print(f"Run name: {RUN_NAME}")
print(f"Work dir: {WORK_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configure GCS credentials

This cell does **not** ask for an upload. It reads credentials in this order:

1. `GOOGLE_APPLICATION_CREDENTIALS`, if already set.
2. A previously-created key file at `/content/gcs_sa.json`, `/kaggle/working/gcs_sa.json`, or the notebook work dir.
3. A `GCS_SA_KEY` Python variable, if you pasted the service-account JSON string into this cell or a cell above it.
4. A Colab Secret named `GCS_SA_JSON`.
5. Application Default Credentials/public bucket access.

If you want to use the teammate key directly, paste it into `GCS_SA_KEY = r'''...'''` in the code cell below. Do not commit or share a notebook containing a real private key.

In [ ]:
# Paste the service-account JSON string here only in your private runtime if you
# do not want to use Colab Secrets or GOOGLE_APPLICATION_CREDENTIALS.
# Example:
# GCS_SA_KEY = r'''{"type":"service_account", ... }'''
GCS_SA_KEY = globals().get("GCS_SA_KEY", "").strip()


def _write_gcs_key_json(raw_key: str, key_path: Path) -> bool:
    if not raw_key:
        return False
    try:
        parsed = json.loads(raw_key)
    except json.JSONDecodeError as exc:
        raise ValueError("GCS_SA_KEY/GCS_SA_JSON is not valid JSON") from exc
    if parsed.get("type") != "service_account" or not parsed.get("client_email"):
        raise ValueError("Credential JSON does not look like a service-account key")
    key_path.write_text(json.dumps(parsed))
    key_path.chmod(0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(key_path)
    print(f"Wrote service-account key to {key_path}")
    return True


def configure_gcs_credentials(work_dir: Path) -> None:
    """Configure credentials without an interactive upload prompt."""
    existing = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if existing and Path(existing).exists():
        print(f"Using GOOGLE_APPLICATION_CREDENTIALS={existing}")
        return

    candidate_paths = [
        work_dir / "gcs_sa.json",
        Path("/content/gcs_sa.json"),
        Path("/kaggle/working/gcs_sa.json"),
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(candidate)
            print(f"Using service-account file: {candidate}")
            return

    key_path = work_dir / "gcs_sa.json"
    if _write_gcs_key_json(GCS_SA_KEY, key_path):
        return

    # Colab Secrets path: store the whole JSON string in a secret named GCS_SA_JSON.
    try:
        from google.colab import userdata
        secret_json = userdata.get("GCS_SA_JSON")
        if _write_gcs_key_json((secret_json or "").strip(), key_path):
            return
    except Exception as exc:
        print(f"Colab secret GCS_SA_JSON not available: {type(exc).__name__}")

    print("No explicit key configured. Trying Application Default Credentials/public access.")


configure_gcs_credentials(WORK_DIR)
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)

# Smoke-test access.
probe = next(client.list_blobs(GCS_BUCKET, prefix=MANIFEST_PREFIX + "/", max_results=1), None)
if probe is None:
    raise RuntimeError(f"No manifests found under gs://{GCS_BUCKET}/{MANIFEST_PREFIX}/")
print(f"GCS access OK. First manifest-like object: gs://{GCS_BUCKET}/{probe.name}")

## 4. Utility functions

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def gcs_uri(blob_name: str) -> str:
    return f"gs://{GCS_BUCKET}/{blob_name}"


def safe_local_name(blob_name: str) -> str:
    return blob_name.replace("/", "__")


def list_manifest_blobs() -> list[str]:
    blobs = []
    prefix = MANIFEST_PREFIX.rstrip("/") + "/"
    for blob in tqdm(client.list_blobs(GCS_BUCKET, prefix=prefix), desc="Listing manifests"):
        if blob.name.endswith("_manifest.json"):
            blobs.append(blob.name)
    blobs = sorted(blobs)
    if not blobs:
        raise RuntimeError(f"No manifest JSON files found under gs://{GCS_BUCKET}/{prefix}")
    return blobs


def load_manifest(blob_name: str) -> dict:
    blob = bucket.blob(blob_name)
    return json.loads(blob.download_as_text())


def make_row_from_manifest(manifest_blob: str, manifest: dict, rng: random.Random) -> dict | None:
    entries = manifest.get("generated", []) or []
    valid_entries = []
    for entry in entries:
        files = entry.get("files", {}) or {}
        if files.get("tampered") and files.get("mask"):
            valid_entries.append(entry)
    if not valid_entries:
        return None

    entry = rng.choice(valid_entries)
    files = entry["files"]
    image_id = manifest.get("image_id") or Path(manifest_blob).name.replace("_manifest.json", "")
    tamper_id = entry.get("tamper_id") or Path(files["tampered"]).stem.replace("_tampered", "")

    image_blob = f"{IMAGE_PREFIX}/{files['tampered']}"
    mask_blob = f"{IMAGE_PREFIX}/{files['mask']}"
    clean_blob = f"{IMAGE_PREFIX}/{manifest.get('clean_file')}" if manifest.get("clean_file") else None

    return {
        "image_id": image_id,
        "tamper_id": tamper_id,
        "alteration": entry.get("alteration"),
        "dom_index": entry.get("dom_index"),
        "dom_type": entry.get("dom_type"),
        "bbox": json.dumps(entry.get("bbox") or entry.get("dst_bbox") or []),
        "manifest_blob": manifest_blob,
        "clean_blob": clean_blob,
        "image_blob": image_blob,
        "mask_blob": mask_blob,
        "image_gcs_uri": gcs_uri(image_blob),
        "mask_gcs_uri": gcs_uri(mask_blob),
    }


def download_blob(blob_name: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    bucket.blob(blob_name).download_to_filename(str(destination))
    return destination


def download_dataset_files(df: pd.DataFrame, cache_dir: Path, max_workers: int = 16) -> pd.DataFrame:
    image_dir = cache_dir / "images"
    mask_dir = cache_dir / "masks"
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    jobs = []
    df = df.copy()
    df["local_image_path"] = [str(image_dir / safe_local_name(x)) for x in df["image_blob"]]
    df["local_mask_path"] = [str(mask_dir / safe_local_name(x)) for x in df["mask_blob"]]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for row in df.itertuples(index=False):
            jobs.append(executor.submit(download_blob, row.image_blob, Path(row.local_image_path)))
            jobs.append(executor.submit(download_blob, row.mask_blob, Path(row.local_mask_path)))
        for future in tqdm(as_completed(jobs), total=len(jobs), desc="Downloading image/mask files"):
            future.result()
    return df


def upload_file_to_gcs(local_path: Path, remote_blob_name: str) -> None:
    bucket.blob(remote_blob_name).upload_from_filename(str(local_path))


def upload_outputs_to_gcs(output_dir: Path, output_prefix: str) -> None:
    files = [path for path in output_dir.rglob("*") if path.is_file()]
    for path in tqdm(files, desc="Uploading outputs to GCS"):
        relative = path.relative_to(output_dir).as_posix()
        upload_file_to_gcs(path, f"{output_prefix.rstrip('/')}/{relative}")
    print(f"Uploaded {len(files)} files to gs://{GCS_BUCKET}/{output_prefix}/")

set_seed(SEED)

## 5. Sample 1,000 IDs and create 70/15/15 splits

The split CSV records both the base `image_id` and selected `tamper_id`, plus GCS URIs for the tampered image and mask.

In [ ]:
rng = random.Random(SEED)
manifest_blobs = list_manifest_blobs()
print(f"Found {len(manifest_blobs):,} manifests")

if len(manifest_blobs) < SAMPLE_SIZE:
    raise ValueError(f"Requested {SAMPLE_SIZE} samples but only found {len(manifest_blobs)} manifests")

selected_manifest_blobs = rng.sample(manifest_blobs, SAMPLE_SIZE)
rows = []
failed_manifests = []

# Loading only the selected manifests is much faster than reading all 30k.
with ThreadPoolExecutor(max_workers=32) as executor:
    future_to_blob = {executor.submit(load_manifest, blob): blob for blob in selected_manifest_blobs}
    for future in tqdm(as_completed(future_to_blob), total=len(future_to_blob), desc="Loading selected manifests"):
        blob = future_to_blob[future]
        try:
            manifest = future.result()
            row = make_row_from_manifest(blob, manifest, rng)
            if row is None:
                failed_manifests.append(blob)
            else:
                rows.append(row)
        except Exception as exc:
            failed_manifests.append(blob)
            print(f"Failed to load {blob}: {exc}")

# Top up if a selected manifest had no valid tamper entry.
if len(rows) < SAMPLE_SIZE:
    already = set(selected_manifest_blobs)
    candidates = [blob for blob in manifest_blobs if blob not in already]
    rng.shuffle(candidates)
    for blob in tqdm(candidates, desc="Topping up invalid manifests"):
        if len(rows) >= SAMPLE_SIZE:
            break
        try:
            manifest = load_manifest(blob)
            row = make_row_from_manifest(blob, manifest, rng)
            if row is not None:
                rows.append(row)
        except Exception:
            continue

if len(rows) != SAMPLE_SIZE:
    raise RuntimeError(f"Could only build {len(rows)} valid rows out of requested {SAMPLE_SIZE}")

sample_df = pd.DataFrame(rows)
sample_df = sample_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n_train = int(SAMPLE_SIZE * SPLIT_RATIOS[0])
n_val = int(SAMPLE_SIZE * SPLIT_RATIOS[1])
n_test = SAMPLE_SIZE - n_train - n_val
sample_df.loc[: n_train - 1, "split"] = "train"
sample_df.loc[n_train : n_train + n_val - 1, "split"] = "val"
sample_df.loc[n_train + n_val :, "split"] = "test"

split_counts = sample_df["split"].value_counts().to_dict()
print(f"Split counts: {split_counts}")
print(sample_df[["split", "image_id", "tamper_id", "alteration", "image_gcs_uri", "mask_gcs_uri"]].head())

split_csv = OUTPUT_DIR / "sampled_1000_splits.csv"
sample_df.to_csv(split_csv, index=False)
print(f"Wrote split/ID record to {split_csv}")

if failed_manifests:
    failed_path = OUTPUT_DIR / "failed_selected_manifests.txt"
    failed_path.write_text("\n".join(failed_manifests))
    print(f"Skipped {len(failed_manifests)} selected manifests; wrote details to {failed_path}")

## 6. Download sampled images and masks to local cache

Training reads local files for speed and to avoid GCS client/thread contention inside PyTorch workers.

In [ ]:
sample_df = download_dataset_files(sample_df, CACHE_DIR, max_workers=24)
indexed_csv = OUTPUT_DIR / "sampled_1000_splits_with_local_paths.csv"
sample_df.to_csv(indexed_csv, index=False)
print(f"Cached {len(sample_df)} examples and wrote {indexed_csv}")

# Quick sanity check that masks are not all empty.
mask_nonzero = []
for mask_path in tqdm(sample_df["local_mask_path"].head(100), desc="Checking first 100 masks"):
    arr = np.array(Image.open(mask_path).convert("L"))
    mask_nonzero.append(int((arr > 0).sum()))
print(f"First 100 masks with tampered pixels: {sum(x > 0 for x in mask_nonzero)}/100")

## 7. Dataset, transforms, model, loss, and metrics

In [ ]:
class TamperSegmentationDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = np.array(Image.open(row.local_image_path).convert("RGB"))
        mask = np.array(Image.open(row.local_mask_path).convert("L"))
        mask = (mask > 0).astype("float32")

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        else:
            mask = mask.permute(2, 0, 1)

        return {
            "image": image.float(),
            "mask": mask.float(),
            "image_id": row.image_id,
            "tamper_id": row.tamper_id,
            "alteration": row.alteration,
        }


def build_transforms(image_size: int):
    train_transform = A.Compose([
        A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.25),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    eval_transform = A.Compose([
        A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR, mask_interpolation=cv2.INTER_NEAREST),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train_transform, eval_transform


class SegformerForgeryDetector(nn.Module):
    def __init__(self, encoder_name=ENCODER_NAME, encoder_weights=ENCODER_WEIGHTS):
        super().__init__()
        self.model = smp.Segformer(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )

    def forward(self, x):
        return self.model(x)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.reshape(-1)
        targets = targets.reshape(-1)
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice


class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        total = self.bce_weight * bce_loss + self.dice_weight * dice_loss
        return total, bce_loss, dice_loss


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0

    def update(self, value, n=1):
        self.sum += float(value) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def batch_metrics(logits, targets, threshold=THRESHOLD, smooth=1.0):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dim=dims)
    pred_sum = preds.sum(dim=dims)
    target_sum = targets.sum(dim=dims)
    union = pred_sum + target_sum - intersection

    dice = ((2.0 * intersection + smooth) / (pred_sum + target_sum + smooth)).mean().item()
    iou = ((intersection + smooth) / (union + smooth)).mean().item()
    pixel_acc = (preds == targets).float().mean().item()
    return dice, iou, pixel_acc


class EarlyStopper:
    def __init__(self, patience=3, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = None
        self.bad_epochs = 0

    def improved(self, value):
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value):
        if self.improved(value):
            self.best = value
            self.bad_epochs = 0
            return False, True
        self.bad_epochs += 1
        return self.bad_epochs >= self.patience, False


train_transform, eval_transform = build_transforms(IMAGE_SIZE)
train_df = sample_df[sample_df.split == "train"].reset_index(drop=True)
val_df = sample_df[sample_df.split == "val"].reset_index(drop=True)
test_df = sample_df[sample_df.split == "test"].reset_index(drop=True)

train_loader = DataLoader(
    TamperSegmentationDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    TamperSegmentationDataset(val_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TamperSegmentationDataset(test_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
print(f"Device: {device}")

## 8. Training and evaluation helpers

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None, epoch=0, phase="train"):
    is_train = optimizer is not None
    model.train(is_train)

    loss_meter = AverageMeter()
    bce_meter = AverageMeter()
    dice_loss_meter = AverageMeter()
    dice_meter = AverageMeter()
    iou_meter = AverageMeter()
    acc_meter = AverageMeter()

    pbar = tqdm(loader, desc=f"{phase.title()} epoch {epoch}", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        batch_size = images.size(0)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=MIXED_PRECISION and device.type == "cuda"):
                logits = model(images)
                loss, bce_loss, dice_loss = criterion(logits, masks)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        with torch.no_grad():
            dice, iou, acc = batch_metrics(logits.detach(), masks.detach())

        loss_meter.update(loss.item(), batch_size)
        bce_meter.update(bce_loss.item(), batch_size)
        dice_loss_meter.update(dice_loss.item(), batch_size)
        dice_meter.update(dice, batch_size)
        iou_meter.update(iou, batch_size)
        acc_meter.update(acc, batch_size)
        pbar.set_postfix(loss=f"{loss_meter.avg:.4f}", dice=f"{dice_meter.avg:.4f}", iou=f"{iou_meter.avg:.4f}")

    return {
        "loss": loss_meter.avg,
        "bce_loss": bce_meter.avg,
        "dice_loss": dice_loss_meter.avg,
        "dice": dice_meter.avg,
        "iou": iou_meter.avg,
        "pixel_acc": acc_meter.avg,
    }


def save_checkpoint(path: Path, model, optimizer, epoch: int, lr: float, metrics: dict, trial_index: int) -> None:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "learning_rate": lr,
        "trial_index": trial_index,
        "metrics": metrics,
        "encoder_name": ENCODER_NAME,
        "encoder_weights": ENCODER_WEIGHTS,
        "image_size": IMAGE_SIZE,
        "threshold": THRESHOLD,
        "sample_ids_csv": str(split_csv),
    }
    torch.save(checkpoint, path)


def train_trial(trial_index: int, learning_rate: float) -> tuple[dict, list[dict]]:
    set_seed(SEED + trial_index)
    model = SegformerForgeryDetector().to(device)
    criterion = CombinedLoss(bce_weight=0.5, dice_weight=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-7
    )
    scaler = torch.cuda.amp.GradScaler(enabled=MIXED_PRECISION and device.type == "cuda")
    early_stopper = EarlyStopper(patience=EARLY_STOP_PATIENCE, min_delta=MIN_DELTA, mode="max")

    trial_dir = CHECKPOINT_DIR / f"trial_{trial_index:02d}_lr_{learning_rate:g}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoint_path = trial_dir / "best.pt"

    best = {
        "trial_index": trial_index,
        "learning_rate": learning_rate,
        "best_epoch": None,
        "best_val_dice": -math.inf,
        "best_val_iou": None,
        "best_val_loss": None,
        "checkpoint_path": str(best_checkpoint_path),
        "stopped_epoch": None,
        "stop_reason": "max_epochs",
    }
    history = []

    print(f"\n=== Trial {trial_index}/3 | learning_rate={learning_rate:g} ===")
    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler, epoch, phase="train")
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None, epoch=epoch, phase="val")
        scheduler.step(val_metrics["loss"])

        record = {
            "trial_index": trial_index,
            "learning_rate": learning_rate,
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "current_lr": optimizer.param_groups[0]["lr"],
        }
        history.append(record)

        should_stop, improved = early_stopper.step(val_metrics["dice"])
        if improved:
            best.update({
                "best_epoch": epoch,
                "best_val_dice": val_metrics["dice"],
                "best_val_iou": val_metrics["iou"],
                "best_val_loss": val_metrics["loss"],
            })
            save_checkpoint(best_checkpoint_path, model, optimizer, epoch, learning_rate, val_metrics, trial_index)
            print(
                f"Trial {trial_index} epoch {epoch}: new best val Dice={val_metrics['dice']:.4f}, "
                f"IoU={val_metrics['iou']:.4f}, loss={val_metrics['loss']:.4f}"
            )
        else:
            print(
                f"Trial {trial_index} epoch {epoch}: val Dice={val_metrics['dice']:.4f}, "
                f"best={best['best_val_dice']:.4f}, bad_epochs={early_stopper.bad_epochs}/{EARLY_STOP_PATIENCE}"
            )

        pd.DataFrame(history).to_csv(trial_dir / "history.csv", index=False)
        if should_stop:
            best["stopped_epoch"] = epoch
            best["stop_reason"] = f"early_stop_patience_{EARLY_STOP_PATIENCE}"
            print(f"Stopping trial {trial_index}: no validation Dice improvement for {EARLY_STOP_PATIENCE} epochs.")
            break

    if best["stopped_epoch"] is None:
        best["stopped_epoch"] = history[-1]["epoch"]
    return best, history

## 9. Run three learning-rate trials

This is the long-running cell. It will stop each LR trial early when validation Dice does not improve for 3 consecutive epochs, or at 50 epochs.

In [ ]:
all_history = []
trial_summaries = []

for trial_index, lr in enumerate(LEARNING_RATE_TRIALS, start=1):
    best, history = train_trial(trial_index, lr)
    trial_summaries.append(best)
    all_history.extend(history)

history_df = pd.DataFrame(all_history)
summary_df = pd.DataFrame(trial_summaries).sort_values("best_val_dice", ascending=False).reset_index(drop=True)

history_path = OUTPUT_DIR / "trial_history.csv"
summary_path = OUTPUT_DIR / "trial_summary.csv"
history_df.to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)

best_trial = summary_df.iloc[0].to_dict()
best_summary_path = OUTPUT_DIR / "best_trial_summary.json"
best_summary_path.write_text(json.dumps(best_trial, indent=2))

print("Trial summary:")
display(summary_df)
print(f"Best LR: {best_trial['learning_rate']} | best epoch: {int(best_trial['best_epoch'])} | val Dice: {best_trial['best_val_dice']:.4f}")
print(f"Wrote {history_path}")
print(f"Wrote {summary_path}")
print(f"Wrote {best_summary_path}")

## 10. Evaluate the best checkpoint on the held-out test split

In [ ]:
def load_best_model(checkpoint_path: str):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = SegformerForgeryDetector(
        encoder_name=checkpoint.get("encoder_name", ENCODER_NAME),
        encoder_weights=None,
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint

best_model, best_checkpoint = load_best_model(best_trial["checkpoint_path"])
criterion = CombinedLoss(bce_weight=0.5, dice_weight=0.5)
test_metrics = run_epoch(best_model, test_loader, criterion, optimizer=None, scaler=None, epoch=int(best_trial["best_epoch"]), phase="test")

test_result = {
    "best_trial": best_trial,
    "test_metrics": test_metrics,
    "test_count": len(test_df),
    "split_csv": str(split_csv),
}
test_metrics_path = OUTPUT_DIR / "test_metrics.json"
test_metrics_path.write_text(json.dumps(test_result, indent=2))

print("Test metrics from best validation checkpoint:")
print(json.dumps(test_metrics, indent=2))
print(f"Wrote {test_metrics_path}")

## 11. Visualize test predictions

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])


def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = np.clip((image * IMAGENET_STD + IMAGENET_MEAN) * 255.0, 0, 255).astype(np.uint8)
    return image


def visualize_predictions(model, dataset, count=6, threshold=THRESHOLD, seed=SEED):
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), min(count, len(dataset)))
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    model.eval()
    for row_idx, ds_idx in enumerate(indices):
        sample = dataset[ds_idx]
        image_tensor = sample["image"]
        mask_tensor = sample["mask"]
        with torch.no_grad():
            logits = model(image_tensor.unsqueeze(0).to(device))
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred = (prob > threshold).astype(np.uint8)
        image = denormalize_image(image_tensor)
        mask = mask_tensor[0].cpu().numpy()

        overlay = image.copy()
        overlay[pred > 0] = (0.55 * overlay[pred > 0] + 0.45 * np.array([255, 0, 0])).astype(np.uint8)

        axes[row_idx, 0].imshow(image)
        axes[row_idx, 0].set_title(f"Image\n{sample['tamper_id']}")
        axes[row_idx, 1].imshow(mask, cmap="gray")
        axes[row_idx, 1].set_title("Ground truth mask")
        axes[row_idx, 2].imshow(prob, cmap="magma", vmin=0, vmax=1)
        axes[row_idx, 2].set_title("Predicted probability")
        axes[row_idx, 3].imshow(overlay)
        axes[row_idx, 3].set_title("Prediction overlay")
        for col in range(4):
            axes[row_idx, col].axis("off")

    fig.tight_layout()
    return fig


test_dataset_for_viz = TamperSegmentationDataset(test_df, eval_transform)
fig = visualize_predictions(best_model, test_dataset_for_viz, count=6)
viz_path = OUTPUT_DIR / "best_model_test_predictions.png"
fig.savefig(viz_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {viz_path}")

## 12. Optional: upload outputs back to GCS

Set `UPLOAD_OUTPUTS_TO_GCS = True` in the config cell and rerun this cell if you want the split IDs, histories, summaries, checkpoints, and visualizations uploaded to:

`gs://maverick91/audit_finding_project/screenshot_tamper_final/segformer_results/<RUN_NAME>/`

In [ ]:
if UPLOAD_OUTPUTS_TO_GCS:
    upload_outputs_to_gcs(OUTPUT_DIR, OUTPUT_GCS_PREFIX)
else:
    print("UPLOAD_OUTPUTS_TO_GCS is False; outputs remain local.")
    print(f"Local outputs: {OUTPUT_DIR}")

## 13. Final result object

This cell prints the key outputs you need to report: best learning rate, best epoch, best validation metric, and held-out test metrics.

In [ ]:
final_result = {
    "run_name": RUN_NAME,
    "sample_size": SAMPLE_SIZE,
    "split_counts": sample_df["split"].value_counts().to_dict(),
    "learning_rate_trials": LEARNING_RATE_TRIALS,
    "max_epochs": MAX_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "best_trial": best_trial,
    "test_metrics": test_metrics,
    "output_dir": str(OUTPUT_DIR),
    "split_ids_csv": str(split_csv),
    "trial_history_csv": str(history_path),
    "trial_summary_csv": str(summary_path),
    "test_metrics_json": str(test_metrics_path),
}
print(json.dumps(final_result, indent=2))